# Build a reproducible experiment

**Time:** 90–120 minutes plus one full pipeline run  
**Prerequisites:** notebooks 04–08.

## Learning objectives

- State one hypothesis and change one variable
- Give every run a unique ID and immutable artifact directory
- Record code revision, environment, model, seed, commands, and file hashes
- Compare experiments using declared success metrics

> A result is reproducible when a teammate can identify exactly what ran and repeat it—not merely when your notebook still has output cells.

In [ ]:
import datetime as dt
import hashlib
import json
import os
import platform
import shlex
import subprocess
import sys
from pathlib import Path

def find_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / 'fanuc_er4ia_sim' / 'run_sort_pipeline.sh').is_file(): return p
    raise FileNotFoundError('Repository root not found')
ROOT = find_root(); SIM = ROOT / 'fanuc_er4ia_sim'; RUNS = SIM / 'runs'
print('Repository:', ROOT)

## 1. Write the experiment card before running

A useful question is narrow and falsifiable: ‘Does adding explicit non-cube exclusions improve six-cube detection completeness on five held-out layouts without increasing JSON failures?’ It is better than ‘Can we improve the AI?’

In [ ]:
EXPERIMENT = {
    'slug': 'baseline-repeat',
    'owner': 'replace-with-name',
    'hypothesis': 'The unchanged pipeline should reproduce six verified pickups and placements.',
    'independent_variable': 'new capture; all configured code and model inputs held constant',
    'controls': ['git revision', 'Cosmos snapshot', 'layout seed', 'camera', 'planner gates'],
    'primary_metric': 'task_complete',
    'acceptance': {'task_complete': True, 'verified_pickups': 6, 'verified_placements': 6},
    'notes': 'Edit this card before creating the run.',
}
print(json.dumps(EXPERIMENT, indent=2))

## 2. Create a unique, empty run directory

Use UTC timestamps so machines agree. Never reuse a directory: overwriting one artifact destroys the chain of evidence.

In [ ]:
stamp = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_ID = f"{stamp}_{EXPERIMENT['slug']}"
RUN = RUNS / RUN_ID
print('Proposed run:', RUN)
assert not RUN.exists(), 'Choose a new ID; experiment directories are immutable.'

## 3. Capture provenance

Record paths but never secrets. A dirty Git tree is not automatically wrong, but the manifest must say it is dirty and later include hashes of the source files that matter.

In [ ]:
def command_output(args):
    try: return subprocess.run(args, cwd=ROOT, text=True, capture_output=True, check=True).stdout.strip()
    except Exception as exc: return f'unavailable: {type(exc).__name__}'

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()

SOURCE_FILES = ['scene_config.py', 'cell.py', 'capture_scene.py', 'perceive.py', 'plan_sort.py', 'sort_scene.py']
manifest = {
    'schema_version': 1, 'run_id': RUN_ID, 'created_utc': stamp, 'experiment': EXPERIMENT,
    'git_commit': command_output(['git', 'rev-parse', 'HEAD']),
    'git_status_porcelain': command_output(['git', 'status', '--porcelain']),
    'host': {'platform': platform.platform(), 'python': sys.version},
    'runtime_paths': {k: os.environ.get(k) for k in ['ISAAC','COSMOS','COSMOS3_NANO_PATH','COSMOS3_ROOT']},
    'gpu': command_output(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader']),
    'source_sha256': {name: sha256(SIM / name) for name in SOURCE_FILES},
}
print(json.dumps(manifest, indent=2)[:5000])

## 4. Save the card, then run the standard pipeline

The opt-in cell creates the run, writes the pre-run manifest, and launches the supported pipeline. The pipeline itself refuses a non-empty directory, so the manifest is temporarily stored beside the run and moved in afterward. This preserves the empty-directory contract.

In [ ]:
MANIFEST_STAGING = RUNS / f'{RUN_ID}.manifest.json'
print('Command:', shlex.join(['bash', str(SIM / 'run_sort_pipeline.sh'), str(RUN)]))
RUN_LIVE = False  # Review the experiment card and paths before opting in.
if RUN_LIVE:
    RUNS.mkdir(parents=True, exist_ok=True)
    if RUN.exists(): raise FileExistsError(RUN)
    MANIFEST_STAGING.write_text(json.dumps(manifest, indent=2) + '\n')
    try:
        subprocess.run(['bash', str(SIM / 'run_sort_pipeline.sh'), str(RUN)], cwd=SIM, check=True)
    finally:
        if RUN.exists() and MANIFEST_STAGING.exists(): MANIFEST_STAGING.replace(RUN / 'experiment_manifest.json')

## 5. Finalize outcome and artifact hashes

A failed experiment still deserves a complete record. Hashing outputs detects accidental edits and makes two copies comparable. Do not include huge frame folders unless you specifically need per-frame integrity.

In [ ]:
if RUN.exists():
    outputs = [p for p in RUN.iterdir() if p.is_file() and p.name != 'experiment_manifest.json']
    manifest['artifact_sha256'] = {p.name: sha256(p) for p in sorted(outputs)}
    report_path = RUN / 'execution_report.json'
    manifest['outcome'] = json.loads(report_path.read_text()) if report_path.exists() else {'task_complete': False, 'reason': 'execution_report.json missing'}
    (RUN / 'experiment_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
    print('Finalized:', RUN / 'experiment_manifest.json')
else: print('Run has not been executed.')

## 6. Compare only compatible experiments

Before comparing metrics, confirm the same task, model snapshot, camera, held-out scenes, and scoring code. If two manifests differ in several causal variables, the comparison is descriptive—not evidence that one specific change helped.

In [ ]:
for path in sorted(RUNS.glob('*/experiment_manifest.json')) if RUNS.exists() else []:
    m = json.loads(path.read_text()); out = m.get('outcome', {})
    print({'run': m.get('run_id'), 'commit': str(m.get('git_commit'))[:10],
           'complete': out.get('task_complete'), 'pickups': out.get('verified_pickups'),
           'placements': out.get('verified_placements'),
           'variable': m.get('experiment', {}).get('independent_variable')})

## Team exercise and checkpoint

Run a baseline repeat and one single-variable experiment. Exchange only the run directories with another teammate. They should be able to identify the hypothesis, revision, runtimes, model path, source hashes, commands, acceptance criteria, and outcome without asking you.

You are finished when the second teammate can reproduce the setup or explain precisely which unavailable external dependency prevents it.